# 🇧🇷 Fine-tuning do BERTimbau em Português

## O que você vai aprender neste notebook?

Este notebook te leva passo a passo pelo processo completo de **fine-tuning** de um modelo BERT em português.

### O que é Fine-tuning?
Fine-tuning é como "ensinar" um modelo já treinado a entender melhor um assunto específico. 
Imagine um professor que já sabe muito, mas você quer que ele se especialize em um tema particular.

### O que vamos fazer?
1. **Carregar dados** - Ler textos em português
2. **Preparar os dados** - Limpar e organizar o texto
3. **Configurar o modelo** - Preparar o BERTimbau com técnicas eficientes (LoRA)
4. **Treinar** - Deixar o modelo aprender com seus dados
5. **Testar** - Ver se o modelo aprendeu corretamente

### Técnicas que usaremos:
- **LoRA**: Técnica que reduz drasticamente o uso de memória
- **MLM**: Masked Language Modeling (o modelo aprende a prever palavras mascaradas)
- **Early Stopping**: Parar o treinamento quando o modelo para de melhorar

Vamos começar!

In [30]:
# ============================================================================
# PASSO 1: Importar Bibliotecas Necessárias
# ============================================================================
# Aqui importamos todas as ferramentas que vamos usar neste notebook.
# Pense nisso como trazer os "ingredientes" antes de começar a cozinhar.

import torch                          # Framework de deep learning
import pandas as pd                   # Para manipular dados em tabelas
import re                             # Para limpeza de texto (regex)
from datasets import Dataset          # Para criar datasets estruturados
from transformers import (
    AutoModelForSequenceClassification,  # Modelo BERT para classificação
    AutoTokenizer,                       # Converte texto em números
    TrainingArguments,                   # Configurações de treinamento
    Trainer,                             # Gerenciador do treinamento
    DataCollatorWithPadding              # Prepara dados para o modelo
)
from peft import LoraConfig, get_peft_model, TaskType  # LoRA para eficiência

# ============================================================================
# PASSO 2: Definir Configurações Globais
# ============================================================================
# Estas são as "receitas" que vamos usar em todo o notebook.
# Você pode mudar estes valores conforme necessário.

# Qual modelo vamos usar? BERTimbau é BERT treinado especificamente para português
MODEL_NAME = 'neuralmind/bert-base-portuguese-cased'

# Onde está o arquivo com o texto que queremos usar para treinar?
BOOK_PATH = './biblia.txt'  # Mude isso para o caminho do seu arquivo

# Onde vamos salvar o modelo treinado?
OUTPUT_DIR = "./results_bertimbau_ft"

print("✅ Bibliotecas importadas com sucesso!")
print(f"📚 Modelo a usar: {MODEL_NAME}")
print(f"📁 Arquivo de entrada: {BOOK_PATH}")
print(f"💾 Modelo será salvo em: {OUTPUT_DIR}")

In [2]:
# ============================================================================
# PASSO 3: Fazer Login no Hugging Face (Opcional)
# ============================================================================
# Se você quer usar modelos privados ou salvar seus modelos no Hub,
# precisa fazer login com sua conta Hugging Face.
# Se não tiver conta, pode pular esta célula.

from huggingface_hub import login

# Descomente a linha abaixo se quiser fazer login
# login()  # Isso abrirá uma janela para você inserir seu token

print("💡 Dica: Se quiser fazer login, descomente a linha acima e execute novamente.")

📖 Etapa 1: Preparação do Dataset e Tokenização
1. Carregamento, Limpeza, **Normalização (Correção)** do Texto
Esta etapa transforma o texto bruto em um dataset formatado com rótulos (labels).

In [31]:
# ============================================================================
# PASSO 4: Carregar, Limpar e Preparar os Dados
# ============================================================================
# Este é um passo MUITO importante. Dados de qualidade = modelo de qualidade.
# Vamos:
# 1. Ler o arquivo de texto
# 2. Limpar o texto (remover números, espaços extras, etc)
# 3. Dividir em frases
# 4. Criar um dataset balanceado (50% bíblico, 50% não-bíblico)
# 5. Dividir em treino e teste

import re
from datasets import Dataset, concatenate_datasets, ClassLabel, Features, Value

# Caminho para o arquivo com textos não-bíblicos (para comparação)
NON_BOOK_PATH = './nao_biblico.txt'

def normalize_text_for_bert(text):
    """
    Função para LIMPAR o texto.
    
    O que ela faz:
    - Remove números de versículos (ex: "1:5" )
    - Remove espaços extras
    - Corrige ortografia antiga para moderna (ex: "Christo" -> "Cristo")
    - Remove caracteres especiais
    """
    # Remove números de versículos
    text = re.sub(r'[\d\s]+:\d+\s*', ' ', text, flags=re.MULTILINE)
    # Remove espaços extras
    text = re.sub(r'\s{2,}', ' ', text).strip()
    
    # Corrige ortografia antiga
    text = text.replace('Christo', 'Cristo').replace('peccado', 'pecado')
    text = text.replace('pae', 'pai').replace('Pae', 'Pai')
    text = text.replace('sciencia', 'ciência')
    text = text.replace('circumcisão', 'circuncisão')
    text = text.replace('hypocrisia', 'hipocrisia')
    text = text.replace('phrophetas', 'profetas')
    text = text.replace('espirito', 'espírito')
    text = text.replace('adopção', 'adoção')
    text = text.replace('elle', 'ele')
    
    return text

def load_and_process_file(file_path, label_id):
    """
    Função para CARREGAR e PROCESSAR um arquivo de texto.
    
    Parâmetros:
    - file_path: caminho do arquivo
    - label_id: 1 para bíblico, 0 para não-bíblico
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            raw_text = f.read()
    except FileNotFoundError:
        raise FileNotFoundError(f"❌ Arquivo '{file_path}' não encontrado!")
    
    # Limpa o texto
    normalized_text = normalize_text_for_bert(raw_text)
    
    # Salva o texto normalizado para referência
    if label_id == 1:
        output_normalized_path = './biblia_normalizada.txt'
        with open(output_normalized_path, 'w', encoding='utf-8') as f:
            f.write(normalized_text)
        print(f"✅ Texto normalizado salvo em '{output_normalized_path}'")
    
    # Divide o texto em frases (por pontuação)
    single_line_text = normalized_text.replace('\n', ' ').strip()
    text_chunks = re.split(r'(?<=[.!?])\s+', single_line_text)
    
    # Filtra frases muito curtas (menos de 20 caracteres)
    processed_chunks = [
        chunk.strip()
        for chunk in text_chunks
        if len(chunk.strip()) > 20
    ]
    
    return Dataset.from_dict({
        'text': processed_chunks,
        'label': [label_id] * len(processed_chunks)
    })

# Carrega os dados
print("📖 Carregando dados...")
biblical_dataset = load_and_process_file(BOOK_PATH, 1)
non_biblical_dataset = load_and_process_file(NON_BOOK_PATH, 0)

# Balanceia os dados (usa a quantidade menor de ambos)
min_samples = min(len(biblical_dataset), len(non_biblical_dataset))
print(f"\n⚖️  Balanceando dados: {min_samples} amostras de cada classe")

biblical_dataset = biblical_dataset.select(range(min_samples))
non_biblical_dataset = non_biblical_dataset.select(range(min_samples))

# Combina os dois datasets
dataset = concatenate_datasets([biblical_dataset, non_biblical_dataset])
dataset = dataset.shuffle(seed=42)  # Embaralha para melhor treinamento

# Converte a coluna 'label' para ClassLabel (necessário para estratificação)
new_features = Features({
    'text': Value('string'),
    'label': ClassLabel(num_classes=2, names=['Não Bíblico', 'Bíblico'])
})
dataset = dataset.cast(new_features)

# Divide em treino (90%) e teste (10%)
train_test_split = dataset.train_test_split(
    test_size=0.1,
    seed=42,
    stratify_by_column="label"  # Garante que ambas as classes estejam em treino e teste
)
train_dataset = train_test_split['train']
eval_dataset = train_test_split['test']

print(f"\n📊 Estatísticas dos dados:")
print(f"   Total de amostras: {len(dataset)}")
print(f"   Treino: {len(train_dataset)} amostras")
print(f"   Teste: {len(eval_dataset)} amostras")
print(f"   Bíblicas no treino: {sum(train_dataset['label'])}")
print(f"   Não-bíblicas no treino: {len(train_dataset) - sum(train_dataset['label'])}")

✅ Texto bíblico normalizado salvo em './biblia_normalizada.txt'

Ambos os datasets serão truncados para 218 amostras (Balanceamento).
Convertendo a coluna 'label' para ClassLabel...


Casting the dataset:   0%|          | 0/436 [00:00<?, ? examples/s]


Total de amostras: 436
Tamanho do treino: 392
Amostras Bíblicas (Label 1) no Treino: 196 / Não Bíblicas (Label 0): 196


📖 Célula 2: Carregamento, Limpeza e Normalização dos Dados (Corrigida)
Esta célula define a função de limpeza e normalização ortográfica e, em seguida, carrega, processa e divide os dados. A lógica de divisão foi ajustada para gerar mais amostras

In [32]:
# ============================================================================
# PASSO 5: Tokenização - Converter Texto em Números
# ============================================================================
# O modelo BERT não entende palavras, entende apenas números.
# Tokenização é o processo de converter texto em números que o modelo entende.
#
# Exemplo:
#   Texto: "Olá mundo"
#   Tokens: [101, 7592, 2088, 102]  (números especiais no início e fim)
#
# O tokenizador também:
# - Divide palavras em partes menores (subwords)
# - Adiciona tokens especiais ([CLS], [SEP], [PAD])
# - Limita o tamanho máximo (256 tokens para BERT)

# Carregar o tokenizador do BERTimbau
print("🔤 Carregando tokenizador...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Garantir que o tokenizador tem um token de padding
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    print("✅ Token de padding adicionado")

def tokenize_function(examples):
    """
    Função para TOKENIZAR o texto.
    
    O que ela faz:
    - Converte texto em números
    - Trunca textos muito longos (máximo 256 tokens)
    - Adiciona padding (preenchimento) para igualar tamanhos
    """
    return tokenizer(
        examples["text"],
        truncation=True,      # Corta textos muito longos
        max_length=256        # Máximo de 256 tokens (BERT padrão é 512)
    )

# Aplicar tokenização a todos os dados
print("\n🔄 Tokenizando dados de treino...")
tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)

print("🔄 Tokenizando dados de teste...")
tokenized_eval_dataset = eval_dataset.map(tokenize_function, batched=True)

# Remover a coluna 'text' original (não precisamos mais dela)
tokenized_train_dataset = tokenized_train_dataset.remove_columns(["text"])
tokenized_eval_dataset = tokenized_eval_dataset.remove_columns(["text"])

print("\n✅ Tokenização concluída!")
print(f"   Exemplo de tokens: {tokenized_train_dataset[0]['input_ids'][:20]}...")

Map:   0%|          | 0/392 [00:00<?, ? examples/s]

Map:   0%|          | 0/44 [00:00<?, ? examples/s]

In [33]:
# ============================================================================
# PASSO 6: Configurar o Modelo com LoRA
# ============================================================================
# LoRA (Low-Rank Adaptation) é uma técnica que torna o fine-tuning MUITO mais eficiente.
#
# Sem LoRA: Precisaríamos treinar 110 MILHÕES de parâmetros (muita memória!)
# Com LoRA: Treinamos apenas ~1 MILHÃO de parâmetros (muito mais rápido e eficiente!)
#
# Como funciona?
# - LoRA adiciona "adaptadores" pequenos ao modelo
# - Estes adaptadores aprendem as mudanças necessárias
# - O modelo original fica congelado (não muda)
# - Resultado: 99% da qualidade com 1% do custo computacional!

print("🤖 Carregando modelo BERTimbau...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,              # 2 classes: Bíblico ou Não-Bíblico
    use_safetensors=True,      # Formato seguro para salvar modelos
    from_tf=False,             # Não usar versão TensorFlow
)

print("\n⚙️  Configurando LoRA...")
lora_config = LoraConfig(
    lora_alpha=16,             # Escala dos pesos LoRA (quanto mais alto, mais forte)
    lora_dropout=0.1,          # Dropout para regularização (evita overfitting)
    r=32,                      # Rank LoRA (dimensão dos adaptadores)
    bias="none",              # Não treinar bias
    task_type=TaskType.SEQ_CLS,  # Tipo de tarefa: Classificação de Sequência
)

# Aplicar LoRA ao modelo
model = get_peft_model(model, lora_config)

print("\n✅ Modelo configurado com LoRA!")
print("\n📊 Estatísticas do modelo:")
model.print_trainable_parameters()
print("\n💡 Explicação:")
print("   - trainable params: Parâmetros que vão ser treinados (LoRA)")
print("   - all params: Total de parâmetros do modelo")
print("   - trainable%: Percentual de parâmetros a treinar (deve ser ~1%)")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



--- BERTimbau configurado com LoRA ---
trainable params: 1,181,186 || all params: 110,105,860 || trainable%: 1.0728


In [36]:
# ============================================================================
# PASSO 7: Configurar e Executar o Treinamento
# ============================================================================
# Agora vamos "ensinar" o modelo com nossos dados.
#
# O que acontece durante o treinamento:
# 1. O modelo recebe um lote de textos
# 2. Faz uma previsão (Bíblico ou Não-Bíblico)
# 3. Compara com a resposta correta
# 4. Calcula o erro (loss)
# 5. Ajusta os pesos LoRA para reduzir o erro
# 6. Repete com o próximo lote
#
# Depois de cada época, testamos com dados que o modelo nunca viu (validação).

print("🔧 Configurando argumentos de treinamento...")

# Data Collator: Prepara os dados para o modelo
# Ele adiciona padding (preenchimento) para igualar tamanhos
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Argumentos de Treinamento
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,              # Onde salvar o modelo
    per_device_train_batch_size=8,      # Quantos textos processar por vez
    gradient_accumulation_steps=2,      # Acumular gradientes (aumenta batch efetivo)
    optim="adamw_torch",               # Otimizador (algoritmo de aprendizado)
    logging_steps=10,                   # Mostrar progresso a cada 10 passos
    learning_rate=5e-5,                 # Taxa de aprendizado (quanto mudar os pesos)
    num_train_epochs=5,                 # Quantas vezes passar pelos dados
    eval_strategy="epoch",              # Avaliar após cada época
    save_strategy="epoch",              # Salvar modelo após cada época
    fp16=True,                          # Usar precisão reduzida (mais rápido)
    dataloader_drop_last=True,          # Descartar último lote incompleto
    load_best_model_at_end=True,        # Carregar melhor modelo ao final
)

print("\n🚀 Inicializando Trainer...")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print("\n⏱️  Iniciando treinamento...")
print("   Isto pode levar alguns minutos. Você verá o progresso abaixo.")
print("   Procure pela métrica 'eval_loss' - quanto menor, melhor!\n")

trainer.train()

print("\n✅ Treinamento concluído!")
print(f"\n💾 Salvando modelo em {OUTPUT_DIR}...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"\n🎉 Sucesso! Modelo e tokenizador salvos em {OUTPUT_DIR}")
print("\n📊 O que foi salvo:")
print("   - adapter_config.json: Configuração do LoRA")
print("   - adapter_model.bin: Pesos do LoRA (pequeno!)")
print("   - tokenizer.json: Tokenizador")
print("   - training_args.bin: Argumentos de treinamento")

/tmp/ipykernel_2332/2584248159.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(



--- Iniciando Treinamento com BERTimbau ---


Epoch,Training Loss,Validation Loss
1,0.473400,0.388364
2,0.356400,0.295624
3,0.288500,0.236931
4,0.239500,0.205457
5,0.239200,0.195799



✅ Fine-Tuning concluído! Modelo e Tokenizador salvos em ./results_bertimbau_ft


In [37]:
# ============================================================================
# PASSO 8: Testar o Modelo Treinado
# ============================================================================
# Agora vamos usar o modelo para fazer previsões em textos novos!
#
# O que vamos fazer:
# 1. Carregar o modelo base + pesos LoRA
# 2. Fundir os pesos (merge) para ter um modelo único
# 3. Fazer previsões em frases de teste
# 4. Ver a confiança do modelo (probabilidade)

from peft import PeftModel
import torch.nn.functional as F

print("🔄 Carregando modelo treinado...")

# Carregar modelo base
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    use_safetensors=True,
    from_tf=False,
)

# Adicionar pesos LoRA
model_to_infer = PeftModel.from_pretrained(base_model, OUTPUT_DIR).eval()

# Mover para GPU se disponível
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_to_infer.to(device)

print(f"✅ Modelo carregado em {device}")

def classificar_texto(texto):
    """
    Função para CLASSIFICAR um texto.
    
    Retorna:
    - Probabilidade de ser Bíblico
    - Probabilidade de ser Não-Bíblico
    """
    # Tokenizar o texto
    inputs = tokenizer(texto, return_tensors="pt", truncation=True, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Fazer previsão
    with torch.no_grad():
        outputs = model_to_infer(**inputs)
    
    # Converter logits em probabilidades
    probabilities = F.softmax(outputs.logits, dim=1)[0]
    
    print(f"\n📝 Texto: {texto}")
    print(f"   Não-Bíblico: {probabilities[0].item():.2%}")
    print(f"   Bíblico: {probabilities[1].item():.2%}")
    
    return probabilities

# Testes
print("\n" + "="*60)
print("TESTANDO O MODELO")
print("="*60)

# Teste 1: Texto bíblico (esperado: alta probabilidade de Bíblico)
print("\n🧪 Teste 1: Texto Bíblico")
texto_biblico = "Disse-lhe Jesus: Eu sou o caminho, e a verdade, e a vida."
classificar_texto(texto_biblico)

# Teste 2: Texto não-bíblico (esperado: alta probabilidade de Não-Bíblico)
print("\n🧪 Teste 2: Texto Não-Bíblico")
texto_secular = "A nova lei tributária será votada na próxima terça-feira pelo congresso."
classificar_texto(texto_secular)

print("\n" + "="*60)
print("✅ Testes concluídos!")
print("="*60)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.



TEXTO: Disse-lhe Jesus: Eu sou o caminho, e a verdade, e a vida.
Prob. Classe 0 (Outro Domínio): 0.2995
Prob. Classe 1 (Domínio Bíblico): 0.7005

TEXTO: A nova lei tributária será votada na próxima terça-feira pelo congresso.
Prob. Classe 0 (Outro Domínio): 0.7951
Prob. Classe 1 (Domínio Bíblico): 0.2049


In [38]:
# Célula de Inferência com Máscara - CORRIGIDA
from transformers import BertForMaskedLM, BertTokenizer
from peft import PeftModel # <--- Importação necessária
import torch
import torch.nn.functional as F

# Usamos o modelo Base e o Diretório de Saída
BASE_MODEL_NAME = "neuralmind/bert-base-portuguese-cased" 
OUTPUT_DIR = "./results_bertimbau_ft" # Deve ser o mesmo de 'Célula 1'
LABELS = {0: "Não Bíblico", 1: "Bíblico"} # <--- VARIÁVEL LABELS DEFINIDA AQUI
import torch
print(f"Versão atual do PyTorch: {torch.__version__}")
try:
    base_model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL_NAME, 
        num_labels=2, 
        use_safetensors=True, 
        ignore_mismatched_sizes=True
    )
    
    # 2. Carregar os pesos LoRA (adaptadores) e fundi-los ao modelo base
    # Isso transforma o 'base_model' no 'modelo_treinado'
    # Nota: Usamos o .eval() e .merge_and_unload() para preparar o modelo para o pipeline de inferência
    model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
    model = model.merge_and_unload() # Fusão dos pesos LoRA ao base
    model.eval()

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    print(f"✅ Modelo ajustado (LoRA fundido) e Tokenizador carregados em {device}.")
    
    # 3. Função de Previsão de Classe
    def classificar_frase(frase):
        # Tokeniza a frase
        inputs = tokenizer(frase, return_tensors="pt", truncation=True, padding=True).to(device)
        
        # Passa pelo modelo e obtém os logits
        with torch.no_grad():
            outputs = model(**inputs)
        
        # Converte logits para probabilidades (Softmax)
        probabilities = F.softmax(outputs.logits, dim=-1)[0]
        
        # Obtém a classe com maior probabilidade
        predicted_class_id = probabilities.argmax().item()
        
        print("-" * 50)
        print(f"FRASE: '{frase}'")
        
        for i, prob in enumerate(probabilities):
            label = LABELS.get(i, f"Label {i}")
            print(f"  {label}: {prob.item() * 100:.2f}%")
        
        print(f"\n-> PREVISÃO FINAL: {LABELS[predicted_class_id]} ({probabilities.max().item() * 100:.2f}%)")
        print("-" * 50)
        return predicted_class_id, probabilities

    # 4. Testes de Inferência
    
    # Exemplo 1: Domínio Bíblico (Esperado: 1 - Bíblico)
    frase_1 = "E o Senhor disse a Moisés: 'Eu farei chover pão dos céus para vós.'"
    classificar_frase(frase_1)
    
    # Exemplo 2: Domínio Não-Bíblico (Esperado: 0 - Não Bíblico)
    frase_2 = "Ele comprou um carro novo na loja da esquina."
    classificar_frase(frase_2)

    # Exemplo 3: Frase com vocabulário antigo/formal (Teste de generalização)
    frase_3 = "Porque a inclinação da carne é morte, mas a inclinação do espírito é vida e paz."
    classificar_frase(frase_3)
    
    # Exemplo 4: Frase neutra ou moderna (Teste de distinção)
    frase_4 = "O estudo da engenharia de software é essencial para a indústria."
    classificar_frase(frase_4)
except Exception as e:
    print(f"\n❌ Erro Crítico após a atualização: {e}")
    print("Verifique se o seu ambiente foi reiniciado após o 'pip install --upgrade torch'.")
    

Versão atual do PyTorch: 2.6.0+cu124


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


✅ Modelo ajustado (LoRA fundido) e Tokenizador carregados em cuda.
--------------------------------------------------
FRASE: 'E o Senhor disse a Moisés: 'Eu farei chover pão dos céus para vós.''
  Não Bíblico: 28.57%
  Bíblico: 71.43%

-> PREVISÃO FINAL: Bíblico (71.43%)
--------------------------------------------------
--------------------------------------------------
FRASE: 'Ele comprou um carro novo na loja da esquina.'
  Não Bíblico: 68.93%
  Bíblico: 31.07%

-> PREVISÃO FINAL: Não Bíblico (68.93%)
--------------------------------------------------
--------------------------------------------------
FRASE: 'Porque a inclinação da carne é morte, mas a inclinação do espírito é vida e paz.'
  Não Bíblico: 29.22%
  Bíblico: 70.78%

-> PREVISÃO FINAL: Bíblico (70.78%)
--------------------------------------------------
--------------------------------------------------
FRASE: 'O estudo da engenharia de software é essencial para a indústria.'
  Não Bíblico: 82.64%
  Bíblico: 17.36%

-> P

In [40]:
# Célula de Inferência de Máscara (fill-mask)

from transformers import BertForMaskedLM, AutoTokenizer, pipeline
from peft import PeftModel
import torch

# 1. Variáveis
OUTPUT_DIR = "./results_bertimbau_ft" # Diretório onde os adaptadores LoRA foram salvos
BASE_MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

# 2. Carregar o Modelo Base como BertForMaskedLM
try:
    # Carrega o modelo base com a 'cabeça' de Masked Language Model (MLM)
    base_model = BertForMaskedLM.from_pretrained(
        BASE_MODEL_NAME, 
        use_safetensors=True, 
        ignore_mismatched_sizes=True # Permite ignorar a 'cabeça' de classificação antiga
    )
    
    # 3. Carrega os adaptadores LoRA salvos do seu fine-tuning de SEQ_CLS
    # e aplica (funde) eles ao BertForMaskedLM base.
    model_peft = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
    model = model_peft.merge_and_unload()
    model.eval()
    
    # Carrega o Tokenizador
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    print(f"✅ Modelo ajustado (LoRA fundido) e Tokenizador carregados em {device}.")
    
    # 4. Criar a pipeline de fill-mask
    unmasker = pipeline(
        "fill-mask",
        model=model,
        tokenizer=tokenizer,
        device=0 if torch.cuda.is_available() else -1
    )
    # 5. Função de Previsão de Máscara
    def prever_palavra_mascarada(frase_mascarada, top_k=5):
        # A pipeline gerencia a substituição de [MASK] pelo token correto
        resultados = unmasker(frase_mascarada, top_k=top_k)
        
        print("\n" + "=" * 50)
        print(f"FRASE COM MÁSCARA: {frase_mascarada}")
        print("=" * 50)
        
        for i, res in enumerate(resultados):
            score = res['score'] * 100
            frase_preenchida = res['sequence']
            
            print(f"  {i+1}. {frase_preenchida} (Probabilidade: {score:.2f}%)")

    # 6. Testes
    
    frase_biblica = "O senhor é meu [MASK], e nada me faltará."
    prever_palavra_mascarada(frase_biblica)
    
    frase_jfa = "A Graça e a [MASK] de nosso Senhor Jesus Christo seja convosco."
    prever_palavra_mascarada(frase_jfa)
    
    frase_neutra = "O [MASK] do Brasil é a capital do país."
    prever_palavra_mascarada(frase_neutra)
    
except Exception as e:
    print(f"\n❌ Erro ao carregar o modelo ajustado: {e}")
    print("Verifique se o diretório OUTPUT_DIR e o modelo base estão corretos.")

Some weights of the model checkpoint at neuralmind/bert-base-portuguese-cased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


✅ Modelo ajustado (LoRA fundido) e Tokenizador carregados em cuda.

FRASE COM MÁSCARA: O senhor é meu [MASK], e nada me faltará.
  1. O senhor é meu amigo, e nada me faltará. (Probabilidade: 23.49%)
  2. O senhor é meu Deus, e nada me faltará. (Probabilidade: 8.53%)
  3. O senhor é meu pai, e nada me faltará. (Probabilidade: 7.38%)
  4. O senhor é meu irmão, e nada me faltará. (Probabilidade: 5.13%)
  5. O senhor é meu Senhor, e nada me faltará. (Probabilidade: 4.91%)

FRASE COM MÁSCARA: A Graça e a [MASK] de nosso Senhor Jesus Christo seja convosco.
  1. A Graça e a Paz de nosso Senhor Jesus Christo seja convosco. (Probabilidade: 72.68%)
  2. A Graça e a Misericórdia de nosso Senhor Jesus Christo seja convosco. (Probabilidade: 13.85%)
  3. A Graça e a paz de nosso Senhor Jesus Christo seja convosco. (Probabilidade: 10.88%)
  4. A Graça e a Glória de nosso Senhor Jesus Christo seja convosco. (Probabilidade: 0.45%)
  5. A Graça e a Vida de nosso Senhor Jesus Christo seja convosco. (Prob